In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/squad-dataset/train.json


In [4]:
import pandas as pd
import numpy as np
import json
import torch
import math

In [5]:
file_path = "/kaggle/input/squad-dataset/train.json"
squad_dict = {}
with open(file_path,'r') as f:
  squad_dict = json.load(f)

In [9]:
min_context_len = 10000;
max_context_len = 0;
average_context_size = 0;
total_number_of_context = 0;
context_greater_than_500 = 0

In [10]:
questionContext = {};
data = squad_dict['data']
row_Data = []  #creating list
context_index  = 0
cnt = 0
for article in data:
    for paragraph in article['paragraphs']:
        context = paragraph['context']

        context_words = context.split()
        min_context_len = min(min_context_len, len(context_words))
        max_context_len = max(max_context_len, len(context_words))
        total_number_of_context += 1
        if len(context_words) > 500:
            context_greater_than_500+=1
        
        for qa in paragraph['qas']:
            if not qa['is_impossible']:
                question = qa['question']
                for ans in qa['answers']:
                    answer = ans['text']
                    start = ans['answer_start']
                    end = start + len(answer)
                    row_Data.append(
                        {
                            'context':context,
                            'question':question,
                            'answer':answer,
                            'start':start,
                            'end':end,
                            'c_id':context_index
                        }
                    )
                    #adding question context
                    questionContext[question] = context_index
        context_index += 1

In [11]:
print("min_context_len : ",min_context_len)
print("max_context_len : ",max_context_len)
print("total_number_of_context : ",total_number_of_context)
print("total_number of context greater than 500 : ", context_greater_than_500)

counter = 0
for i in questionContext:
  counter+=1;
print("size of question Context : ", counter)

min_context_len :  20
max_context_len :  653
total_number_of_context :  19035
total_number of context greater than 500 :  3
size of question Context :  86769


In [19]:
row = pd.DataFrame(row_Data)

In [20]:

l = row['context'].tolist()
documents = row[['context','c_id']].drop_duplicates().reset_index(drop=True)
documents

,context,c_id
0,Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...,0
1,Following the disbandment of Destiny's Child i...,1
2,"A self-described ""modern-day feminist"", Beyonc...",2
3,"Beyoncé Giselle Knowles was born in Houston, T...",3
4,Beyoncé attended St. Mary's Elementary School ...,4
...,...,...
18877,"Institute of Medicine, the central college of ...",18975
18878,Football and Cricket are the most popular spor...,18976
18879,The total length of roads in Nepal is recorded...,18977
18880,The main international airport serving Kathman...,18978


In [21]:
!pip install transformers faiss-cpu datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 56.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 11.8 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
bigframes 1.42.0 requires rich<14,>=12.4.4, but you have rich 14.0.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.9.0.13 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cudnn-cu12==9.1.0.70; platform_system == "Linux" and platform_machine == "x86_64", but you have

In [28]:
import faiss
model_name = "intfloat/e5-small-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval().cuda()

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 384, padding_idx=0)
    (position_embeddings): Embedding(512, 384)
    (token_type_embeddings): Embedding(2, 384)
    (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=384, out_features=384, bias=True)
            (key): Linear(in_features=384, out_features=384, bias=True)
            (value): Linear(in_features=384, out_features=384, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=384, out_features=384, bias=True)
            (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [29]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

In [30]:

import torch.nn.functional as F
from tqdm import tqdm


# Mean Pooling Function
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state 
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    pooled = torch.sum(token_embeddings * input_mask_expanded, dim=1) / input_mask_expanded.sum(dim=1)
    return pooled

# Encode Single Batch of Texts
def encode_text(texts):
    encoded_input = tokenizer(texts, return_tensors='pt', padding=True, truncation=True, max_length=512).to(device)

    with torch.no_grad():
        model_output = model(**encoded_input)
    
    sentence_embeddings = mean_pooling(model_output, encoded_input['attention_mask'])
    sentence_embeddings = F.normalize(sentence_embeddings, p=2, dim=1)  # L2 normalize

    return sentence_embeddings.detach().cpu().numpy()


def encode_in_batches(texts, batch_size=32):
    all_embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Encoding"):
        batch_texts = texts[i:i + batch_size]
        batch_embeddings = encode_text(batch_texts)
        all_embeddings.append(batch_embeddings)
    return np.vstack(all_embeddings)


In [31]:
texts = documents['context'].tolist()
doc_embeddings = encode_in_batches(texts, batch_size=32)

Encoding: 100%|██████████| 591/591 [01:47<00:00,  5.48it/s]


In [32]:
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)

In [35]:
question = []
for keys, value in questionContext.items():
  question.append(keys)

In [39]:
def dcg(scores):
    """Compute Discounted Cumulative Gain"""
    return sum([rel / math.log2(idx + 2) for idx, rel in enumerate(scores)])


def compute_metrics(question, questionContext, index, k=10, max_samples=2000):
    total_correct = 0
    reciprocal_ranks = []
    ndcg_scores = []
    recall_scores = []
    precision_scores = []
    mar_scores = []

    for idx, ques in enumerate(question):
        if idx >= max_samples:
            break

        query_embedding = encode_text([ques])
        distances, indices = index.search(query_embedding, k)

        predictions = indices[0]
        actual_index = questionContext.get(ques, -1)

        # Relevance array: 1 at correct prediction position, 0 otherwise
        relevance = [1 if pred == actual_index else 0 for pred in predictions]

        # Accuracy@k
        if actual_index in predictions:
            total_correct += 1

        # MRR
        if actual_index in predictions:
            rank = list(predictions).index(actual_index)
            reciprocal_ranks.append(1.0 / (rank + 1))
        else:
            reciprocal_ranks.append(0.0)

        # NDCG
        ideal_relevance = sorted(relevance, reverse=True)
        ndcg = dcg(relevance) / dcg(ideal_relevance) if sum(ideal_relevance) > 0 else 0.0
        ndcg_scores.append(ndcg)

        # Recall@k and Precision@k
        recall = 1.0 if actual_index in predictions else 0.0
        precision = 1.0 / (list(predictions).index(actual_index) + 1) if actual_index in predictions else 0.0
        recall_scores.append(recall)
        precision_scores.append(precision)

        # MAR (Mean Average Relevance)
        mar = sum(relevance) / k
        mar_scores.append(mar)

    num_samples = min(len(question), max_samples)
    return {
        f"Accuracy@{k}": (total_correct / num_samples) * 100,
        f"MRR@{k}": np.mean(reciprocal_ranks),
        f"NDCG@{k}": np.mean(ndcg_scores),
        f"Recall@{k}": np.mean(recall_scores),
        f"Precision@{k}": np.mean(precision_scores),
        f"MAR@{k}": np.mean(mar_scores)
    }



🔍 Top-1 Evaluation Metrics:
Accuracy@1: 68.8400
MRR@1: 0.6884
NDCG@1: 0.6884
Recall@1: 0.6884
Precision@1: 0.6884
MAR@1: 0.6884

🔍 Top-2 Evaluation Metrics:
Accuracy@2: 81.7600
MRR@2: 0.7530
NDCG@2: 0.7699
Recall@2: 0.8176
Precision@2: 0.7530
MAR@2: 0.4088

🔍 Top-3 Evaluation Metrics:
Accuracy@3: 86.8200
MRR@3: 0.7699
NDCG@3: 0.7952
Recall@3: 0.8682
Precision@3: 0.7699
MAR@3: 0.2894

🔍 Top-4 Evaluation Metrics:
Accuracy@4: 89.7000
MRR@4: 0.7771
NDCG@4: 0.8076
Recall@4: 0.8970
Precision@4: 0.7771
MAR@4: 0.2243

🔍 Top-5 Evaluation Metrics:
Accuracy@5: 91.8000
MRR@5: 0.7813
NDCG@5: 0.8157
Recall@5: 0.9180
Precision@5: 0.7813
MAR@5: 0.1836

🔍 Top-10 Evaluation Metrics:
Accuracy@10: 95.8200
MRR@10: 0.7870
NDCG@10: 0.8291
Recall@10: 0.9582
Precision@10: 0.7870
MAR@10: 0.0958


In [ ]:
samples = [2000, 5000, 10000, 20000, 86769]

final_result = []

for spl in samples:
    temp = []
    for k in [1, 2, 3, 4, 5, 10, 15, 20, 25, 30, 40, 50, 75, 100]:
        results = compute_metrics(
            question = question,
            questionContext=questionContext,
            index = index,
            k=k,
            max_samples=spl
        )
        temp.append(results)
        print(f"\n🔍 Top-{k} Evaluation Metrics:")
        for metric, value in results.items():
            print(f"{metric}: {value:.4f}")
    final_result.append(temp)

In [ ]:
# saving Data for further calculation
with open("final_result.json", "w") as f:
    json.dump(final_result, f)